# 실험 02: 프롬프트 템플릿화 (PromptTemplate)

## 1. 실험 제목과 목표
- **제목**: 프롬프트의 동적 조립과 재사용성 확보
- **목표**: 프롬프트를 매번 하드코딩하는 대신 `PromptTemplate`과 `ChatPromptTemplate`을 사용해 변수화(빈칸 뚫기)하는 방법을 배우고, 늦은 바인딩(Partial Variables) 기법을 알아봅니다.

## 2. 실험 개요
1. **비교 실험 1**: Python 기본 `f-string` vs LangChain `PromptTemplate` 비교
2. **비교 실험 2**: `ChatPromptTemplate`을 활용한 멀티 롤(System/User) 템플릿화
3. **비교 실험 3**: `partial_variables`를 활용한 부분 변수 주입(지연 바인딩)
4. **실패/주의 케이스**: 선언된 변수를 입력하지 않았을 때 발생하는 Missing 변수 에러 시연

## 3. 라이브러리 import

In [1]:
import os
from datetime import datetime
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

## 4. 환경 변수 로드 및 모델 준비

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 5. 비교 실험 1: f-string vs PromptTemplate
프롬프트 안에 들어갈 내용(예: 검색 키워드)이 계속 바뀔 때 어떻게 관리할까요?

In [3]:
product = "아이폰 15"
tone = "유머러스하게"

print("=== [1. 파이썬 f-string 포매팅] ===")
f_string_prompt = f"{product}에 대한 광고 문구를 {tone} 작성해줘."
print("생성된 프롬프트:", f_string_prompt)
# f-string은 프롬프트 문자열과 파이썬 코드가 강하게 결합되어 관리가 어렵습니다.

print("\n=== [2. LangChain PromptTemplate] ===")
# 템플릿 객체 생성 (변수는 {중괄호}로 표시)
template = PromptTemplate.from_template("{product}에 대한 광고 문구를 {tone} 작성해줘.")

# 변수를 주입(.format)하여 문자열 완성
lc_prompt_str = template.format(product="갤럭시 S24", tone="진지하게")
print("생성된 프롬프트:", lc_prompt_str)

print("\n-> 결과: PromptTemplate을 쓰면 프롬프트 텍스트 자체를 외부에 파일(JSON/YAML)로 빼두고 변수만 주입하는 등 유연한 관리가 가능해집니다.")

=== [1. 파이썬 f-string 포매팅] ===
생성된 프롬프트: 아이폰 15에 대한 광고 문구를 유머러스하게 작성해줘.

=== [2. LangChain PromptTemplate] ===
생성된 프롬프트: 갤럭시 S24에 대한 광고 문구를 진지하게 작성해줘.

-> 결과: PromptTemplate을 쓰면 프롬프트 텍스트 자체를 외부에 파일(JSON/YAML)로 빼두고 변수만 주입하는 등 유연한 관리가 가능해집니다.


## 6. 비교 실험 2: ChatPromptTemplate
실제 모델들은 System 역할과 User 역할을 나누어 받습니다. 여러 역할을 조합하는 템플릿입니다.

In [4]:
# 메시지들의 리스트 형태로 템플릿을 정의합니다.
chat_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {profession} 전문가입니다. 전문 용어를 섞어서 답변하세요."),
    ("user", "{topic}에 대해 설명해주세요.")
])

# 변수 주입
formatted_messages = chat_template.format_messages(profession="의학", topic="면역력 강화")

print("[주입된 메시지 객체들]")
for msg in formatted_messages:
    print(f"{type(msg).__name__} -> {msg.content}")

print("\n[LLM 실제 호출 결과]")
res_chat = llm.invoke(formatted_messages)
print(res_chat.content[:200] + "...")

[주입된 메시지 객체들]
SystemMessage -> 당신은 의학 전문가입니다. 전문 용어를 섞어서 답변하세요.
HumanMessage -> 면역력 강화에 대해 설명해주세요.

[LLM 실제 호출 결과]
면역력 강화는 신체의 면역 시스템을 최적화하여 병원체에 대한 방어력을 높이는 과정을 의미합니다. 면역 시스템은 비특이적 면역과 특이적 면역으로 나눌 수 있으며, 두 가지 모두 중요한 역할을 합니다.

1. **영양소 섭취**: 비타민 C, 비타민 D, 아연 및 셀레늄과 같은 항산화제가 풍부한 음식을 섭취하면 면역세포의 기능을 강화할 수 있습니다. 특히, 프...


In [5]:
# 메시지들의 리스트 형태로 템플릿을 정의합니다.
chat_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {profession} 전문가입니다. 전문 용어를 섞어서 답변하세요."),
    ("user", "{topic}에 대해 설명해주세요.")
])

# 변수 주입
formatted_messages = chat_template.format_messages(profession="법률", topic="뺑소니")

print("[주입된 메시지 객체들]")
for msg in formatted_messages:
    print(f"{type(msg).__name__} -> {msg.content}")

print("\n[LLM 실제 호출 결과]")
res_chat = llm.invoke(formatted_messages)
print(res_chat.content[:200] + "...")

[주입된 메시지 객체들]
SystemMessage -> 당신은 법률 전문가입니다. 전문 용어를 섞어서 답변하세요.
HumanMessage -> 뺑소니에 대해 설명해주세요.

[LLM 실제 호출 결과]
뺑소니는 교통사고의 일종으로, 교통사고를 일으킨 후 피해자에게 필요한 조치를 취하지 않고 도주하는 행위를 말합니다. 우리나라에는 특정범죄 가중처벌 등에 관한 법률 제5조의2에 명시되어 있으며, 이는 고의 또는 과실로 인한 교통사고가 발생했음에도 불구하고, 피해자를 구호하거나 경찰에 보고하지 않고 현장을 이탈하는 경우에 해당합니다.

뺑소니는 도주행위 그 자...


## 7. 비교 실험 3: Partial Variables (지연 바인딩)
변수가 2개인데, 하나는 프로그램 시작 시점에 알고 있고 나머지 하나는 사용자에게 입력받을 때 유용합니다.

In [6]:
def get_today_date():
    return datetime.now().strftime("%Y년 %m월 %d일")

# 템플릿 생성: {date}와 {question} 두 개의 변수가 필요합니다.
time_template = PromptTemplate(
    template="오늘은 {date}입니다. 질문에 답해주세요: {question}",
    input_variables=["question"],            # 나중에 주입할 변수
    partial_variables={"date": get_today_date} # 지금 미리 채워둘 변수 (함수도 가능!)
)

# 이제 'question' 변수 하나만 넣어도 프롬프트가 완성됩니다.
final_prompt_str = time_template.format(question="내일은 며칠인가요?")

print("[Partial Variable 적용 결과]")
print(final_prompt_str)
print("\n[LLM 호출]")
print(llm.invoke(final_prompt_str).content)

[Partial Variable 적용 결과]
오늘은 2026년 06월 27일입니다. 질문에 답해주세요: 내일은 며칠인가요?

[LLM 호출]
내일은 2026년 06월 28일입니다.


## 8. 실패/주의 케이스: 누락된 변수(Missing Variable)
템플릿에 정의된 필수 변수를 주입하지 않으면 어떻게 될까요?

In [9]:
print("[변수 누락 에러 시연]")
strict_template = PromptTemplate.from_template("{name}님, 당신의 직업인 {job}에 대해 말해주세요.")

try:
    # 'job' 변수를 일부러 빼먹었습니다.
    strict_template.format(name="김철수")
except Exception as e:
    print("🔥 에러 발생:", type(e).__name__)
    print("에러 메시지:", str(e))
    print("\n-> 주의: LangChain의 PromptTemplate은 변수 검증(Validation)을 매우 깐깐하게 수행합니다. 필요한 빈칸을 모두 채우지 않으면 바로 KeyError를 발생시켜 런타임 사고를 예방합니다.")

[변수 누락 에러 시연]
🔥 에러 발생: KeyError
에러 메시지: 'job'

-> 주의: LangChain의 PromptTemplate은 변수 검증(Validation)을 매우 깐깐하게 수행합니다. 필요한 빈칸을 모두 채우지 않으면 바로 KeyError를 발생시켜 런타임 사고를 예방합니다.


## 9. 결과 해석

1. **재사용성**: 템플릿을 한 번 만들어 두면 파라미터(변수)만 교체하여 수많은 문장을 공장처럼 찍어낼 수 있습니다.
2. **구조화(ChatPromptTemplate)**: `System`과 `User` 역할을 템플릿 레벨에서부터 하드코딩해 두어, 모델이 맥락을 절대 잊어버리지 않게 강제합니다.
3. **안전성**: 파이썬 `f-string`은 런타임에 유연하지만 에러를 잡기 힘든 반면, `PromptTemplate`은 변수 누락을 엄격하게 감시하여 버그를 줄여줍니다.

## 10. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `f-string`으로 작성할 때보다 코드가 약간 길어보일 수 있지만, {변수}를 템플릿 객체로 관리하니 입력값 검증과 에러 핸들링(`KeyError`)이 매우 명확해짐.
* `partial_variables`에 오늘 날짜를 구하는 '함수' 자체를 넘기면, 프롬프트가 실행되는 시점의 최신 날짜가 자동으로 주입된다는 강력한 장점을 확인함.

### 다음 개선 방향
* 지금까지는 `llm.invoke(template.format(...))` 방식으로 프롬프트와 모델을 수동으로 이어주었음.
* 응답 결과 역시 `res.content`로 수동 추출하고 있는데, 이를 OutputParser를 도입하여 깔끔하게 정리해볼 차례임.